# 추론 모델 밑바닥부터 1편 실습 노트북 — 2장: 사전학습된 LLM으로 글 생성하기 (한국어 주석판)

**출처:** 원본 노트북 `ch02_main.ipynb`, Sebastian Raschka, https://github.com/rasbt/reasoning-from-scratch , Apache License 2.0. 책 *Build a Reasoning Model (From Scratch)*, Manning.

**변경 사항:** 코난쌤(@conanssam)이 설명 글을 한국어로 옮기고 코드에 한국어 주석을 덧붙였습니다. 코드 동작은 원본과 같습니다.

**쓰는 법**
1. 저장소를 내려받습니다: `git clone https://github.com/rasbt/reasoning-from-scratch.git`
2. 저장소 폴더에서 `uv sync`를 실행해 필요한 패키지를 설치합니다.
3. `uv run jupyter lab`으로 주피터랩을 열고, 이 파일을 `ch02/01_main-chapter-code/` 폴더에 넣어 실행합니다. 모델 가중치는 같은 폴더의 `qwen3/` 폴더에 내려받아집니다.

**맥 팁:** 처음에는 원본 권장대로 CPU로 돌려 보세요. 맥 GPU를 쓰고 싶으면 `device = get_device()`가 고른 결과(MPS)를 그대로 쓰면 됩니다.


<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a>의 책 <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>에 딸린 실습 코드입니다<br>
<br>코드 저장소: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 2장: 사전학습된 LLM으로 글 생성하기

이 노트북에서 쓰는 패키지입니다.

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

# 설치된 패키지 버전을 하나씩 출력합니다
for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F01_raschka.webp?1" width="500px">

&nbsp;
## 2.1 글을 만드는 LLM 소개

- 이 절에는 코드가 없습니다.
- LLM은 어떻게 글을 만들어 낼까요?
- 이 장은 준비하는 장입니다. 코딩 환경을 갖추고, 책 전체에서 쓸 LLM을 준비합니다.
- 다음 장들에서 쓰고 넓혀 갈 글 생성(text generation) 함수도 직접 짭니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F02_raschka.webp?1" width="300px">

- LLM(그리고 신경망) 흐름도는 보통 위에서 아래로 그리고, 위에서 아래로 읽습니다.

&nbsp;
## 2.2 코딩 환경 준비하기

- 이 책을 읽고 계시다면 아마 파이썬으로 코딩해 보셨을 거예요.
- 파이썬 환경(파이썬 3.10 이상)이 이미 있다면, 의존성(dependency)을 설치하는 가장 쉬운 방법은 `pip`입니다.

In [ ]:
#!pip install -r https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/refs/heads/main/requirements.txt

- 이 장에 필요한 패키지만 직접 설치할 수도 있습니다.

In [ ]:
#!pip install torch>=2.10.0 tokenizers>=0.22.2 reasoning-from-scratch

- 저는 많이 추천받는 파이썬 패키지·프로젝트 관리 도구 [uv](https://docs.astral.sh/uv/)를 더 좋아합니다.
- `uv`는 공식 사이트에서 내 운영체제에 맞는 설치 방법을 따라 설치하세요: https://docs.astral.sh/uv/getting-started/installation/
- 그다음 깃허브 저장소를 내려받습니다(clone).

In [ ]:
#!git clone --depth 1 https://github.com/rasbt/reasoning-from-scratch.git

- `git`이 없다면 Manning 웹사이트에서 소스 코드 저장소를 받거나, 이 링크를 눌러 직접 받아도 됩니다: https://github.com/rasbt/reasoning-from-scratch/archive/refs/heads/main.zip (받은 뒤 압축을 풀어 주세요)

- 터미널에서 `reasoning-from-scratch` 폴더로 이동합니다.
- 저장소에는 `.python-version` 파일이 들어 있어서, `uv`가 기본으로 파이토치와 맞는 파이썬 버전을 씁니다.
- `uv run jupyter lab`을 실행해 주피터랩을 열고, 빈 노트북이나 이 장의 노트북을 엽니다.
- 이 명령은 로컬 가상 환경(virtual environment, 보통 `.venv/`)도 만들고, `reasoning-from-scratch` 폴더의 `pyproject.toml` 파일에 적힌 의존성을 모두 자동으로 설치합니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F03_raschka.webp?1" width="500px">

- 설치에 대한 더 자세한 내용과 다른 방법이 필요하면 [../02_setup-tips/python-instructions.md](../02_setup-tips/python-instructions.md)를 보세요.

&nbsp;
## 2.3 하드웨어 요구 사항과 추천

- 파이토치(PyTorch)가 처음이라면 제 튜토리얼 [PyTorch in One Hour: From Tensors to Training Neural Networks on Multiple GPUs](https://sebastianraschka.com/teaching/pytorch-1h/)를 먼저 읽어 보시길 권합니다.
- 앞 절을 따라오셨다면 파이토치가 설치되어 있을 거예요.
- 내 파이토치가 GPU를 지원하는지 직접 확인해 봅시다. 내 컴퓨터에서 무엇을 쓸 수 있는지 볼 수 있습니다.

In [ ]:
import torch


print(f"PyTorch version {torch.__version__}")

# 어떤 GPU를 쓸 수 있는지 차례로 확인합니다 (NVIDIA → 인텔 → 애플 실리콘 → CPU)
if torch.cuda.is_available():
    print(f"CUDA/ROCm GPU: {torch.cuda.get_device_name(0)}")

elif torch.xpu.is_available():
    print(f"Intel GPU: {torch.xpu.get_device_name(0)}")

elif torch.backends.mps.is_available():
    print("Apple Silicon GPU")

else:
    print("Only CPU")

- 장에 따라 다르지만, 코드는 NVIDIA(CUDA) GPU가 있으면 자동으로 그것을 쓰고, 없으면 CPU에서 돌아갑니다(어떤 절이나 장에서 권장하는 경우에는 애플 실리콘 GPU를 씁니다).
- 2~4장은 CPU로도 적당한 시간 안에 돌릴 수 있습니다.
- 5~7장 코드는 CPU에서 아주 느립니다. 이 장들에는 CUDA를 지원하는 GPU를 권장합니다(정확히 얼마나 필요한지는 해당 장에서 다룹니다).
- 저는 [Lightning AI Studio](https://lightning.ai/)를 좋아합니다. 가입하고 인증을 마치면 무료 컴퓨팅 크레딧을 줍니다. [Google Colab](https://colab.research.google.com/)도 좋은 선택입니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F04_raschka.webp" width="500px">

- 클라우드 컴퓨팅 추천이 필요하면 [../02_setup-tips/gpu-instructions.md](../02_setup-tips/gpu-instructions.md)를 보세요.
- 하지만 지금은 아직 GPU가 필요 없습니다. 앞쪽 장들은 GPU 없는 컴퓨터에서도 잘 돌아갑니다.

&nbsp;
## 2.4 LLM에 넣을 입력 글 준비하기

- 이 절에서는 토크나이저(tokenizer) 쓰는 법을 배웁니다. 토크나이저로 입력 글을 토큰 ID(token ID)로 바꿔서(인코딩, encode) LLM에 넣습니다.
- 반대로 LLM이 내놓은 결과를 사람이 읽을 수 있는 글로 되돌릴(디코딩, decode) 때도 토크나이저를 씁니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F05_raschka.webp?1" width="500px">

- 앞에서 말했듯이, LLM과 토크나이저를 밑바닥부터 구현하는 것은 이 책의 범위가 아닙니다. 이 책은 이미 있는 LLM과 토크나이저 위에 추론(reasoning) 방법을 밑바닥부터 구현하는 데 집중합니다.
- 이 책에서는 다음 절에서 불러올 사전학습된(pre-trained) LLM을 씁니다. 여기서는 그 LLM과 짝이 되는 토크나이저를 불러옵니다.
- 기본 LLM과 짝 토크나이저를 담은 `reasoning_from_scratch` 파이썬 패키지를 준비해 두었습니다. 토크나이저는 [`tokenizers`](https://github.com/huggingface/tokenizers) 파이썬 라이브러리의 도움을 받아 짰습니다.
- `reasoning_from_scratch` 패키지 코드는 이 책의 실습 코드에 들어 있고, 2.2절 안내를 따랐다면 이미 설치되어 있을 거예요.

- 이제 토크나이저 파일을 내려받습니다(Qwen3 기본 LLM용 토크나이저입니다. 자세한 건 다음 절에서 다룹니다).

In [ ]:
from reasoning_from_scratch.qwen3 import download_qwen3_small

# 토크나이저 파일만 qwen3/ 폴더에 내려받습니다 (모델 가중치는 아직)
download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

- 이제 토크나이저 파일에서 설정을 읽어 `Qwen3Tokenizer`에 불러옵니다.

In [ ]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
# 내려받은 설정 파일로 토크나이저를 만듭니다
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

- 아직 LLM 자체는 불러오지 않았으니, 간단히 한 바퀴만 돌려 봅니다. 글을 토큰 ID로 인코딩한 뒤, 다시 글로 디코딩합니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F06_raschka.webp" width="500px">

In [ ]:
prompt = "Explain large language models."
# 글 → 토큰 ID 숫자 목록 (인코딩)
input_token_ids_list = tokenizer.encode(prompt)

In [ ]:
# 토큰 ID 하나하나가 어떤 글 조각인지 확인합니다
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

In [ ]:
# 토큰 ID 목록 → 원래 글 (디코딩)
text = tokenizer.decode(input_token_ids_list)
print(text)

- `Qwen3Tokenizer`에는 서로 다른 토큰이 약 15만 1천 개 있습니다(어휘 크기, vocabulary size).

- 토큰화(tokenization)를 더 공부할 자료:
  - [Build a Large Language Model (from Scratch)](https://mng.bz/M96o) 2장
  - [Implementing A Byte Pair Encoding (BPE) Tokenizer From Scratch](https://sebastianraschka.com/blog/2025/bpe-from-scratch.html)

&nbsp;
## 2.5 사전학습된 모델 불러오기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F07_raschka.webp" width="500px">

- 앞 절에서 토크나이저를 불러올 때 살짝 말했듯이, 이 책은 Qwen3 0.6B를 씁니다. 어떤 오픈 웨이트(open-weight) 기본 모델을 쓸지 오래 고민한 끝에 Qwen3를 골랐습니다. 이유는 이렇습니다.
  - 이 글을 쓰는 시점에 모델 성능으로 보면 Qwen3가 오픈 웨이트 모델 중 선두입니다.
  - Qwen3 0.6B는 Llama 3 1B보다 메모리를 적게 씁니다.
  - 기본 모델(base model, 추론 모델을 만들 때 주로 쓰는 쪽)과 공식 추론 모델 버전이 둘 다 있어서, 추론 버전을 비교 기준(reference model)으로 쓸 수 있습니다.
- (정식 표기는 "Qwen3"에는 띄어쓰기가 없고 "Llama 3"에는 띄어쓰기가 있습니다.)
- '밑바닥부터'라는 취지에 맞게, 외부 LLM 라이브러리 없이 순수 파이토치로 제가 다시 구현한 Qwen3를 씁니다. 이 구현은 원래 Qwen3 모델 가중치(weight)와 호환됩니다.
- 다만 이 책에서는 Qwen3 구현 코드를 하나하나 살펴보지 않습니다. 그것만으로 책 한 권이 되기 때문입니다(제 책 [Build A Large Language Model (From Scratch)](https://github.com/rasbt/LLMs-from-scratch)처럼요). 대신 이 책(Build A Reasoning Model From Scratch)은 기본 모델(여기서는 Qwen3) 위에 추론 방법을 밑바닥부터 구현하는 데 집중합니다.
- Qwen3 모델 코드는 부록 C에 있습니다.
- 추론 버전과 더 큰 Qwen3 모델을 불러오는 법은 부록 D를 보세요.
- (훨씬) 더 자세한 내용은 Qwen3 [깃허브 저장소](https://github.com/QwenLM/Qwen3)와 [기술 보고서](https://arxiv.org/abs/2505.09388)를 보세요.

- 이 모델은 일반 소비자용 컴퓨터에서도 돌아가도록 일부러 작게 골랐습니다(그래도 꽤 똑똑합니다).
- CPU, NVIDIA GPU(`"cuda"`), 애플 실리콘 GPU(`"mps"`), 인텔 GPU(`"xpu"`)에서 잘 돌아갑니다. 성능을 어떻게 맞바꾸는지는 이 장 뒤쪽에서 다룹니다.

In [ ]:
def get_device(enable_tensor_cores=True):
    # 쓸 수 있는 가장 빠른 장치를 고릅니다. 맥에서는 MPS(애플 실리콘 GPU)를 고릅니다
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")
        
        # NVIDIA GPU에서 행렬 곱을 더 빠른 TF32 정밀도로 하도록 설정합니다
        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            # PyTorch 2.9 and 2.10 still read the legacy TF32 setting in torch.compile.
            # See https://github.com/pytorch/pytorch/issues/166387
            # and https://github.com/rasbt/reasoning-from-scratch/issues/256
            if (major, minor) >= (2, 11):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

# 맥 GPU를 쓰려면 아래 셀의 CPU 고정 줄을 건너뛰고 이 결과를 그대로 쓰면 됩니다
device = get_device()

- 처음 한 번 따라 해 볼 때는 `"cpu"`로 돌리기를 권합니다. 그래서 아래에서 장치(device)를 직접 정해 둡니다.

In [ ]:
# Recommended: Use CPU on the first run-through
# 처음엔 CPU로 고정합니다 (결과가 책과 가장 비슷하게 나옵니다)
device = torch.device("cpu")

- 그다음 사전학습된 모델 가중치가 담긴 파일을 내려받습니다. 크기는 약 1.5GB입니다.

In [ ]:
# 모델 가중치 파일(약 1.5GB)을 qwen3/ 폴더에 내려받습니다
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

- LLM 구조를 아는 분들을 위해 지금 불러오는 Qwen3 0.6B 모델의 구조를 아래에 보여 드립니다. 하지만 이 책에서는 이 구조를 이해하는 게 꼭 필요하거나 중요하지 **않습니다**. 뒤 장들에서 구조를 고치지 않고, 그 위에 추론 기법을 더하기 때문입니다.

- 이 코드 저장소에 들어 있는 [reasoning-from-scratch](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3.py) 파이썬 패키지용으로 Qwen3 모델 구조를 밑바닥부터 짰습니다. 소스 코드는 부록 C에도 실었습니다. 다시 말하지만 궁금한 분을 위한 보너스일 뿐이고, 책의 나머지를 따라가는 데 이 내부를 보거나 이해할 필요는 없습니다.

In [ ]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"

# 설정값으로 빈 모델 뼈대를 만들고
model = Qwen3Model(QWEN_CONFIG_06_B)
# 내려받은 학습된 가중치를 채워 넣습니다
model.load_state_dict(torch.load(model_path))

# 모델을 고른 장치(CPU/GPU)로 옮깁니다
model.to(device)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F08_raschka.webp" width="300px">

&nbsp;
## 2.6 LLM이 글을 차례로 만드는 과정 이해하기

- 이 절에서는 LLM으로 글을 만들 수 있게 간단한 감싸기 함수(wrapper function)를 짭니다(4장에서 기능을 더 붙입니다).

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F09_raschka.webp?1" width="500px">

- LLM은 한 번에 한 단어씩 만들어 냅니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F10_raschka.webp?2" width="500px">

- 위 그림은 새로 만든 단어만 보여 주는 단순화한 그림입니다. 아래 그림은 첫 번째 반복을 확대해서 보여 줍니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp" width="3b00px">

In [ ]:
example = torch.tensor([1, 2, 3]) 
print(example)
# unsqueeze(0): 맨 앞에 차원을 하나 추가 → 모양 [3]이 [1, 3]이 됩니다 (배치 차원)
print(example.unsqueeze(0))

In [ ]:
example = torch.tensor([[1, 2, 3]]) 
print(example)
# squeeze(0): 맨 앞의 크기 1짜리 차원을 없앱니다 → [1, 3]이 [3]으로
print(example.squeeze(0))

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp?2" width="300px">

In [ ]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_list)}")

input_tensor = torch.tensor(input_token_ids_list)
# 모델은 배치 차원이 있는 입력을 받으므로 unsqueeze(0)로 감싸고 장치로 옮깁니다
input_tensor_fmt = input_tensor.unsqueeze(0).to(device)

# 추론 모드: 학습용 기울기 계산을 꺼서 더 빠르고 메모리를 덜 씁니다
with torch.inference_mode():
    output_tensor = model(input_tensor_fmt)

# 출력 모양은 [토큰 수, 어휘 크기]: 위치마다 다음 토큰 후보 약 15만 개의 점수
output_tensor_fmt = output_tensor.squeeze(0)
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F12_raschka.webp" width="500px">

In [ ]:
# 마지막 위치의 점수만 꺼냅니다. 이게 '다음에 올 토큰' 예측입니다
last_token = output_tensor_fmt[-1]
print(last_token)

In [ ]:
# argmax: 점수가 가장 큰 칸의 번호(= 가장 그럴듯한 다음 토큰 ID)를 돌려줍니다
print(torch.argmax(last_token, dim=-1, keepdim=True))

In [ ]:
# 위에서 나온 토큰 ID를 글로 바꿔 봅니다
print(tokenizer.decode([20286]))

In [ ]:
example = torch.tensor([-2, 1, 3, 1])
# max는 가장 큰 값(3)을, argmax는 그 값의 위치(2)를 돌려줍니다
print(torch.max(example))
print(torch.argmax(example))

&nbsp;
## 2.7 가장 간단한 글 생성 함수 짜기


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F13_raschka.webp" width="500px">

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F14_raschka.webp?2" width="500px">

- `generate_text_basic_stream` 함수가 이 차례차례 글을 만드는 과정을 구현합니다.

In [ ]:
@torch.inference_mode()
def generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens, 
    eos_token_id=None
):
    model.eval()

    # 새 토큰을 하나씩, 최대 max_new_tokens개까지 만듭니다
    for _ in range(max_new_tokens):
        # 지금까지의 글 전체를 매번 모델에 다시 넣고, 마지막 위치 점수만 씁니다
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        # Stop if we encounter an end-of-sequence token
        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        # 만든 토큰을 바로 내보내서 실시간으로 출력할 수 있게 합니다
        yield next_token  # Yield each token as it's generated
        
        # 새 토큰을 입력 뒤에 붙여 다음 반복의 입력으로 씁니다
        token_ids = torch.cat([token_ids, next_token], dim=1)

- 간단한 프롬프트(prompt) `"Explain large language models in a single sentence."`로 토큰 100개짜리 답을 만들어 보면서 어떻게 돌아가는지 봅시다(추론 부분은 뒤 장에서 다룹니다).
- 아래 코드는 느립니다. 컴퓨터에 따라 1~3분 걸릴 수 있습니다(뒤 절에서 빠르게 만듭니다).

In [ ]:
prompt = "Explain large language models in a single sentence."
# 프롬프트를 토큰 ID로 바꾸고 배치 차원을 더한 텐서로 만듭니다
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)
max_new_tokens = 100


# 토큰이 하나 나올 때마다 글로 바꿔 바로 화면에 찍습니다
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True  # Deactivates buffering so tokens are printed live
    )

- LLM이 지시를 꽤 잘 따르지만, `<|endoftext|>` 뒤로는 답이 엉뚱하거나 주제에서 벗어납니다. 이 토큰은 학습 때 서로 다른 문서 사이를 나누는 구분자로 쓰입니다.
- LLM을 쓸 때는 이 토큰을 만나면 글 생성을 멈추게 해야 합니다.

In [ ]:
# 문서 끝 표시 토큰의 ID를 확인합니다
print(tokenizer.encode("<|endoftext|>"))

- 편하게 쓰도록 이 토큰 ID는 토크나이저 속성(attribute)으로 저장되어 있습니다(eos = end of sequence, 문장 끝).

In [ ]:
print(tokenizer.eos_token_id)

- 이 값을 써서 LLM(정확히는 `generate_text_basic_stream` 함수)에게 언제 글 생성을 멈출지 알려 줄 수 있습니다.

In [ ]:
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    # 이번엔 문서 끝 토큰이 나오면 멈추도록 알려 줍니다
    eos_token_id=tokenizer.eos_token_id  # Use EOS token
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

- 위 답은 CPU에서 코드를 돌렸을 때 나오는 결과입니다. 장치에 따라 만들어지는 글이 조금 다를 수 있습니다.

- 이 절을 마무리하고 코드를 빠르게 만드는 법을 보기 전에, 계산 성능을 재는 간단한 벤치마크(benchmark) 함수를 만들어 둡니다.

In [ ]:
import warnings

def generate_stats(output_token_ids, tokenizer, start_time,
                   end_time):
    # tokenizer is currently unused but retained for backward compatibility
    # 걸린 시간과 초당 토큰 수(속도)를 출력합니다
    total_time = end_time - start_time
    print(f"\n\nTime: {total_time:.2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} tokens/sec")

    # CUDA·인텔 GPU를 쓸 때는 최대 메모리 사용량도 출력합니다
    for name, backend in (("CUDA", getattr(torch, "cuda", None)),
                          ("XPU", getattr(torch, "xpu", None))):
        if backend is not None and backend.is_available():

            # Check whether we are actually using this backend
            device_type = output_token_ids.device.type
            if device_type != name.lower():
                warnings.warn(
                    f"{name} is available but tensors are on "
                    f"{device_type}. Memory stats may be 0."
                )
    
            # Synchronize if supported (important for async backends)
            if hasattr(backend, "synchronize"):
                backend.synchronize()
            
            max_mem_bytes = backend.max_memory_allocated()
            max_mem_gb = max_mem_bytes / (1024 ** 3)
            print(f"Max {name} memory allocated: {max_mem_gb:.2f} GB")
            backend.reset_peak_memory_stats()

In [ ]:
import time

# 시작 시각을 기록합니다
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

# 모은 토큰을 하나로 합쳐 속도를 계산합니다
output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

&nbsp;
## 2.8 KV 캐시로 추론 빠르게 하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F15_raschka.webp?2" width="500px">

- 이 책의 코드는 읽기 쉬운 쪽을 우선합니다. 최적화만으로도 책 한 권을 따로 쓸 수 있습니다.
- 여기서는 'KV 캐시(KV caching)'라는 엔지니어링 요령을 살펴봅니다(KV는 LLM 어텐션 메커니즘(attention mechanism) 안의 키(key)와 값(value)을 뜻합니다).
- 이 용어가 낯설어도 괜찮습니다. 반복할 때마다 다시 쓰는 중간값을 저장해 둘(캐시) 방법이 있다는 것만 알면 됩니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F16_raschka.webp" width="500px">

- KV 캐시가 돌아가는 원리를 더 자세히 알고 싶다면 제 글 [Understanding and Coding the KV Cache in LLMs from Scratch](https://magazine.sebastianraschka.com/p/coding-the-kv-cache-in-llms)를 보세요.
- 아래는 `generate_text_basic_stream` 함수를 KV 캐시를 쓰도록 고친 버전입니다.

In [ ]:
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_basic_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):
    model.eval()
    # KV 캐시: 앞에서 계산한 키·값을 저장해 두고 다음 토큰 때 다시 씁니다
    cache = KVCache(n_layers=model.cfg["n_layers"])  # New
    model.reset_kv_cache()                           # New

    # 프롬프트 전체는 처음 한 번만 모델에 넣습니다
    out = model(token_ids, cache=cache)[:, -1]
    for step in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token
        if step + 1 < max_new_tokens:  # Skips the forward pass after the last token is generated
            # 그다음부터는 새 토큰 하나만 넣습니다. 앞부분 계산은 캐시가 대신해서 빨라집니다
            out = model(next_token, cache=cache)[:, -1]

- 쓰는 법은 앞과 비슷합니다.

In [ ]:
start_time = time.time()
generated_ids = []

# 캐시 버전 함수로 바꾼 것 말고는 앞과 같습니다
for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

- 보다시피 맥 미니 M4 CPU에서 KV 캐시를 쓰면 글 생성이 훨씬 빨라집니다.

&nbsp;
## 2.9 파이토치 모델 컴파일로 추론 빠르게 하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F17_raschka.webp?2" width="500px">

- 모델 추론(글 생성)을 크게 빠르게 하는 또 하나의 기법은 `torch.compile`입니다.
- 쓰는 법은 간단합니다. 모델에 `torch.compile`을 부르기만 하면 됩니다(다른 옵션은 [공식 문서](https://docs.pytorch.org/docs/stable/torch.compiler_api.html)를 보세요).

In [ ]:
major, minor = map(int, torch.__version__.split(".")[:2])
if (major, minor) >= (2, 8):
    # This avoids retriggering model recompilations 
    # in PyTorch 2.8 and newer
    # if the model contains code like self.pos = self.pos + 1
    # PyTorch 2.8 이상에서 불필요한 재컴파일을 막는 설정입니다
    torch._dynamo.config.allow_unspec_int_on_nn_module = True

# 모델을 컴파일합니다. 실제 컴파일은 처음 실행할 때 일어납니다
model_compiled = torch.compile(model)

# If you have issues with torch.compile on "mps" devices and get an InductorError,
# make sure you are using PyTorch 2.9 or newer

---

**윈도우 참고 1**

- 윈도우에서는 컴파일이 까다로울 수 있습니다.
- `torch.compile()`은 Inductor를 씁니다. Inductor는 커널(kernel)을 실행 중에 컴파일(JIT 컴파일)하므로, 제대로 동작하는 C/C++ 도구 모음(toolchain)이 필요합니다.
- CUDA에서는 Inductor가 Triton도 필요로 합니다. 커뮤니티 패키지 `triton-windows`로 설치할 수 있습니다.
  - `cl not found`가 보이면 ["C++ 워크로드"를 넣어 Visual Studio Build Tools를 설치하고](https://learn.microsoft.com/en-us/cpp/build/vscpp-step-0-installation?view=msvc-170), "x64 Native Tools" 프롬프트에서 파이썬을 실행하세요.
  - CUDA에서 `triton not found`가 보이면 `triton-windows`를 설치하세요(예: `uv pip install "triton-windows<3.4"`).
- CPU의 경우, 한 독자가 이 [윈도우용 PyTorch Inductor 가이드](https://docs.pytorch.org/tutorials/unstable/inductor_windows.html)를 따르라고 추천해 주었습니다.
  - 이때 Visual Studio 2022를 설치하면서 영어 언어 팩을 꼭 함께 설치해야 UTF-8 오류를 피할 수 있습니다.
  - 또 코드를 노트북이 아니라 "Visual Studio 2022 Developer Command Prompt"에서 실행해야 합니다.
- 설정이 너무 까다로우면 컴파일은 건너뛰어도 됩니다. **컴파일은 선택 사항이고, 모든 예제는 컴파일 없이도 잘 돌아갑니다.**

**윈도우 참고 2**

- 독자들 말에 따르면 윈도우에서 기본 설정으로 `torch.compile`을 돌리면 빨라지지 않았습니다. 하지만 `"max-autotune"` 모드로 돌리니 2배 빨라졌다고 합니다: `torch.compile(model, mode="max-autotune")`

---

- 첫 번째 실행은 처음 컴파일과 최적화를 하느라 조금 느릴 수 있습니다. 그래서 글 생성을 여러 번 되풀이합니다.
- 먼저 캐시를 쓰지 않는 버전부터 봅시다(조금 느릴 수 있고 xx분 걸릴 수도 있습니다).

In [ ]:
# 3번 돌립니다. 첫 번째는 컴파일하느라 느린 '워밍업'이라 속도 비교에서 뺍니다
for i in range(3):

    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream(
        # 컴파일한 모델을 씁니다 (캐시 없는 버전)
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # Collect generated tokens
    
    end_time = time.time()
    

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

    print(f"\n{30*'-'}\n")

- 보다시피 초당 5토큰으로, 앞(초당 4토큰)보다 아주 조금 빨라졌을 뿐입니다.
- 이제 KV 캐시 버전은 얼마나 잘하는지 봅시다.

In [ ]:
# 이번엔 컴파일한 모델 + KV 캐시 조합입니다. 역시 첫 번째는 워밍업입니다
for i in range(3):
    
    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream_cache(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # Collect generated tokens
    
    end_time = time.time()

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(
        output_token_ids_tensor, tokenizer, start_time, end_time
    )

    print(f"\n{30*'-'}\n")

- 보다시피 컴파일 덕분에 2배나 빨라졌습니다(초당 64토큰 대 30토큰).
- 아래 표에 결과를 더 정리했습니다.

| 모델       | 방식              | 하드웨어             | 초당 토큰 수  | GPU 메모리 (VRAM) |
|------------|-------------------|----------------------|---------------|-------------------|
| Qwen3Model | 기본              | Mac Mini M4 CPU      | 5             | -                 |
| Qwen3Model | 기본 + 컴파일     | Mac Mini M4 CPU      | 5             | -                 |
| Qwen3Model | KV 캐시           | Mac Mini M4 CPU      | 29            | -                 |
| Qwen3Model | KV 캐시 + 컴파일  | Mac Mini M4 CPU      | 68            | -                 |
|            |                   |                      |               |                   |
| Qwen3Model | 기본              | Mac Mini M4 GPU      | 27            | -                 |
| Qwen3Model | 기본 + 컴파일     | Mac Mini M4 GPU      | 43            | -                 |
| Qwen3Model | KV 캐시           | Mac Mini M4 GPU      | 41            | -                 |
| Qwen3Model | KV 캐시 + 컴파일  | Mac Mini M4 GPU      | 71            | -                 |
|            |                   |                      |               |                   |
| Qwen3Model | 기본              | NVIDIA H100 GPU      | 51            | 1.55 GB           |
| Qwen3Model | 기본 + 컴파일     | NVIDIA H100 GPU      | 164           | 1.81 GB           |
| Qwen3Model | KV 캐시           | NVIDIA H100 GPU      | 48            | 1.52 GB           |
| Qwen3Model | KV 캐시 + 컴파일  | NVIDIA H100 GPU      | 141           | 1.81 GB           |
|            |                   |                      |               |                   |
| Qwen3Model | 기본              | NVIDIA DGX Spark GPU | 74            | 1.53 GB           |
| Qwen3Model | 기본 + 컴파일     | NVIDIA DGX Spark GPU | 103           | 1.49 GB           |
| Qwen3Model | KV 캐시           | NVIDIA DGX Spark GPU | 68            | 1.47 GB           |
| Qwen3Model | KV 캐시 + 컴파일  | NVIDIA DGX Spark GPU | 98            | 1.47 GB           |

- 위의 NVIDIA DGX Spark는 GB10(Blackwell) GPU를 씁니다.
- 모든 예제는 프롬프트 하나로 돌렸습니다(배치 크기(batch size) 1). 여러 프롬프트를 한 번에 처리하는 배치 추론이 궁금하면 부록 E를 보세요.

&nbsp;
## 요약

- 이 절에는 코드가 없습니다.